# Citadel T1C — Large batched learning discriminator (one session)
Preregistered: `docs/citadel/experiments/T1C/PLAN.md`. Four frozen arms (A control / B answer-objective / C narrow-data / D scale) on ~100 MB unique synthetic arithmetic, one session, no interaction between arms. Run cells 0–F in order with no edits. No secrets.

In [ ]:
# 0. Fresh Citadel checkout + pinned read-only Cymek runtime (public clone, no credentials)
import os, subprocess, sys
repo = '/content/An-Ra-colab'
if not os.path.isdir(os.path.join(repo, '.git')):
    subprocess.run(['git','clone','--depth','50','-b','citadel','https://github.com/dhurv0045com-spec/An-Ra-the-new-AGI.git',repo], check=True)
else:
    subprocess.run(['git','-C',repo,'fetch','origin','citadel','--depth','50'], check=True)
    subprocess.run(['git','-C',repo,'checkout','citadel'], check=True)
    subprocess.run(['git','-C',repo,'reset','--hard','origin/citadel'], check=True)
%cd /content/An-Ra-colab
os.environ['CITADEL_ROOT'] = repo
os.environ.setdefault('PJRT_DEVICE', 'TPU')
os.environ['CITADEL_PLATFORM'] = 'colab'
if repo not in sys.path: sys.path.insert(0, repo)
from citadel_tpu import runtime_bootstrap as rb
rt_root, rt_sha = rb.ensure_cymek_runtime()
print('CITADEL_SHA=' + str(rb.citadel_sha()))
print('CYMEK_RUNTIME_SHA=' + str(rt_sha))
SESSION = 'docs/citadel/tpu_receipts/t1c_session'
print('SESSION_DIR=' + SESSION)

In [ ]:
# A. T1C preflight gate (determinism, specs, budgets, disk, TPU, XLA APIs). If NO, STOP.
import subprocess
pa = subprocess.run(['python','-m','citadel_tpu.t1c_preflight'])
assert pa.returncode == 0, 'T1C PREFLIGHT failed — READY_FOR_T1C=NO. Diagnose, do not run.'
print('READY_FOR_T1C=YES')

In [ ]:
# B. Throughput calibration: picks ONE static shape for all arms (recorded, reused on resume).
from citadel_tpu import t1c_run as t1c
cal = t1c.calibrate(out=SESSION + '/THROUGHPUT_CALIBRATION.json')
print('selected:', cal['selected'], f"{cal['selected_tokens_per_second']:.0f} tok/s")

In [ ]:
# C. Generate + manifest the ~100 MB corpus (streamed, O(1) memory; reused on resume).
from citadel_tpu import arith_data as ad
man = ad.build_manifest(out=SESSION + '/DATA_MANIFEST.json')
print('bytes:', man['total_bytes'], 'leakage:', man['leakage'])

In [ ]:
# D. Execute the entire matrix (arms A→D, resume-safe, per-arm isolation). No interaction needed.
session = t1c.run_session(SESSION)
print(session['arms'], session['labels'])

In [ ]:
# E. Cross-arm summary (machine-evaluated preregistered rules; read, do not reinterpret).
import json
cs = json.load(open(SESSION + '/CROSS_ARM_SUMMARY.json'))
print('labels:', cs['labels'])
print(json.dumps(cs['reasons'], indent=2))

In [ ]:
# F. Export one bundle (+ checkpoint binaries separately if the notebook lists them).
import json
from google.colab import files
files.download(SESSION + '/CITADEL_T1C_RESULTS.zip')
bm = json.load(open(SESSION + '/BUNDLE_MANIFEST.json'))
print('zip bytes:', bm['zip_bytes'], 'checkpoints bundled:', bm['checkpoints_bundled'])
if not bm['checkpoints_bundled']:
    for p in bm['checkpoints']:
        files.download(p)
print('transfer the bundle back to the operator')